In [1]:
import numpy as np
import scipy.constants as const
import uncertainties as unc
import uncertainties.unumpy as up
import matplotlib.pyplot as plt
import pandas as pd
from numpy import linalg
%matplotlib inline

In [2]:
def lin_fit(t_data, y_data, freq, plot=True, resid=True, raw=False, lbound=0, ubound=-1):
    """Function which takes in data and linearly fits it using least squares method. Assumes three coefficients such that 
    y=a*sin(wt) + b*cos(wt) + c is the solution.

    Returns coefficients and fit arrays in a tuple of the structure (list of coefficients, ([t_data, fit solution], residual)).

    Parameters:
    freq: (int or float) estimated frequency of function in Hz (if time data is in seconds)
    plot: (boolean) toggles whether to show plots or not
    resid: (boolean) toggles whether to show residuals or not
    raw: (boolean) toggles whether to plot raw data or not
    lbound: (int) sets lower bound of plots
    ubound: (int) sets upper bound of plots
    """

    if ubound != -1 or lbound != 0:
        t_data = t_data[lbound:ubound]
        y_data = y_data[lbound:ubound]

    # Create empty matrix for the transpose of x
    xT = [[0], [0], [0]]
    xT[0]=np.sin(2*np.pi*freq*t_data)
    xT[1]=np.cos(2*np.pi*freq*t_data)
    xT[2]=np.ones(np.shape(t_data))

    # Get x by taking the transpose
    x=np.transpose(xT)
    xTxn1=np.linalg.inv(np.matmul(xT,x))

    # # Check shape
    # print(np.shape(xTxn1)) # S.B. 3 x 3 matrix
    xTy=np.matmul(xT, y_data)

    # Coefficients (S.B. 2 x 1 matrix)
    omega = np.matmul(xTxn1, xTy)
    a=omega[0]
    b=omega[1]
    c=omega[2]
    sol = a*np.sin(2*np.pi*freq*t_data) + b*np.cos(2*np.pi*freq*t_data) + c

    # Calculating residuals
    eps = y_data-sol

    if plot:
        if resid:

            # Plot raw data
            fig1, ax1 = plt.subplots(figsize=(16,6))
            if raw:
                ax1.plot(t_data, y_data, 'o', markersize=1, alpha=0.5,label='raw data')
                ax1.legend()

            # Plot fit

            ax1.plot(t_data, sol, '-', label='fit')
            ax1.set_xlabel("Time (s)")
            ax1.set_ylabel("")



            # Plot residuals
            # Figure setup
            fig2, (ax2, ax3) = plt.subplots(nrows=1, ncols=2, figsize=(16,7), sharey=True, gridspec_kw={'width_ratios': [2,1]})
            fig2.subplots_adjust(wspace=0.05)

            # Plot scatter
            ax2.plot(t_data, eps, 'o', markersize=2)
            ax2.set_xlabel("Time (s)")
            ax2.set_ylabel(r"$y-\widetilde{y}$")
            ax2.hlines(0, xmin=np.min(t_data), xmax=np.max(t_data), colors='tab:gray', linestyles='dashed')

            # Plot histogram
            ax3.hist(eps, orientation='horizontal', bins=50)
            ax3.set_xscale('log')
            ax3.set_xlabel("Counts")


        else:
            # Plot just the fit, no residuals
            plt.figure(figsize=(16,5))
            plt.plot(t_data, sol, '-', label='fit')

            # Plot raw data
            if raw:
                plt.plot(t_data, y_data, 'o', markersize=1,label='raw data')
                plt.legend()

    print(f'Coefficients: a={a:.5f}, b={b:.5f}, c={c:.5f}')

    return [a,b,c], ([t_data, sol], eps);


In [3]:

def lin_fit_adv(t_data, y_data, freq, plot=True, resid=True, raw=False, lbound=0, ubound=-1):
    """Function which takes in data and linearly fits it using least squares method. Assumes three coefficients such that
    y=a*sin(wt) + b*cos(wt) + c*sin(2wt) + d*cos(2wt) + e*sin(3wt) + f*cos(3wt) + g is the solution.

    Returns coefficients and fit arrays in a tuple of the structure (list of coefficients, ([t_data, fit solution], residual)).

    Parameters:
    freq: (int or float) estimated frequency of function in Hz (if time data is in seconds)
    plot: (boolean) toggles whether to show plots or not
    resid: (boolean) toggles whether to show residuals or not
    raw: (boolean) toggles whether to plot raw data or not
    lbound: (integer) sets lower bound of plots
    ubound: (integer) sets upper bound of plots
    """

    if ubound != -1 or lbound != 0:
        t_data = t_data[lbound:ubound]
        y_data = y_data[lbound:ubound]

    # Create empty matrix for the transpose of x (one row for each unknown)
    xT = [[0], [0], [0], [0], [0], [0], [0]]
    xT[0]=np.sin(2*np.pi*freq*t_data)
    xT[1]=np.cos(2*np.pi*freq*t_data)
    xT[2]=np.sin(2*np.pi*2*freq*t_data)
    xT[3]=np.cos(2*np.pi*2*freq*t_data)
    xT[4]=np.sin(2*np.pi*3*freq*t_data)
    xT[5]=np.cos(2*np.pi*3*freq*t_data)
    xT[6]=np.ones(np.shape(t_data))

    # Get x by taking the transpose
    x=np.transpose(xT)
    xTxn1=np.linalg.inv( np.matmul(xT,x))

    # # Check shape
    # print(np.shape(xTxn1)) # S.B. 3 x 3 matrix
    xTy=np.matmul(xT, y_data)

    # Coefficients (S.B. 2 x 1 matrix)
    omega = np.matmul(xTxn1, xTy)
    a=omega[0]
    b=omega[1]
    c=omega[2]
    d=omega[3]
    e=omega[4]
    f=omega[5]
    g=omega[6]
    sol = a*np.sin(2*np.pi*freq*t_data) + b*np.cos(2*np.pi*freq*t_data) +c*np.sin(2*np.pi*2*freq*t_data) + d*np.cos(2*np.pi*2*freq*t_data) +e*np.sin(2*np.pi*3*freq*t_data) + f*np.cos(2*np.pi*3*freq*t_data) + g


    # Calculating residuals
    eps = y_data-sol

    if plot:
        if resid:

            # Plot raw data
            fig1, ax1 = plt.subplots(figsize=(16,6))

            if raw:
                ax1.plot(t_data, y_data, 'o', markersize=1, alpha=0.5,label='raw data')


            # Plot fit
            ax1.plot(t_data, sol, '-', label='fit')
            ax1.set_xlabel("Time (s)")
            ax1.set_ylabel("")
            ax1.legend()

            # Plot residuals
            ## Figure setup
            fig2, (ax2, ax3) = plt.subplots(nrows=1, ncols=2, figsize=(16,7), sharey=True, gridspec_kw={'width_ratios': [2,1]})
            fig2.subplots_adjust(wspace=0.05)

            ## Plot scatter
            ax2.plot(t_data, eps, 'o', markersize=2)
            ax2.set_xlabel("Time (s)")
            ax2.set_ylabel(r"$y-\widetilde{y}$")
            ax2.hlines(0, xmin=np.min(t_data), xmax=np.max(t_data), colors='tab:gray', linestyles='dashed')

            ## Plot histogram
            ax3.hist(eps, orientation='horizontal', bins=50)
            ax3.set_xscale('log')
            ax3.set_xlabel("Counts")

        else:
            # Plot just the fit, no residuals
            plt.figure(figsize=(16,5))
            plt.plot(t_data, sol, '-', label='fit')

            # Plot raw data
            if raw:
                plt.plot(t_data, y_data, 'o', markersize=1,label='raw data')
                plt.legend()

    print(f'Coefficients: a={a:.5f}, b={b:.5f}, c={c:.5f}, d={d:.5f}, e={e:.5f}, f={f:.5f}, g={g:.5f}')

    return ([a,b,c,d,e,f,g], ([t_data, sol], eps));

